In [1]:
from transformers import pipeline
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
from nltk import download
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm
from collections import Counter
import pandas as pd
import re, os
import hashlib
import plotly.express as px
import matplotlib.pyplot as plt
import math
tqdm.pandas()
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

d:\Limit\Intric\scraper\hxrscraper\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# EXTRACT SOURCE, TARGET, RELATION, AND FULL_TEXT

import ast

project_name = "pasal_33_uud"
file_name = f"prep_{project_name}-tweets.csv"

base_folder = f"../../result/{project_name}/tweets"
input_folder = f"{base_folder}/preprocessing"

output_folder = f"{base_folder}/sna"
os.makedirs(output_folder, exist_ok=True)

df = pd.read_csv(f"{input_folder}/{file_name}")

def to_plain(val):
    """Convert stringified list ke plain text, e.g. \"['a','b']\" -> \"a b\" """
    if pd.isna(val) or val == '' or val == '-':
        return ''
    val = str(val).strip()
    if val.startswith('['):
        try:
            return ' '.join(ast.literal_eval(val))
        except Exception:
            return val
    return val

edges = []

for _, row in df.iterrows():
    src = row['username']
    exc = ['grok']
    normal_text = row.get('normalize')
    stemm_text = to_plain(row.get('stemming'))
    sentiment = row.get('sentiment')

    # === REPLIED / RETWEETED / QUOTED ===
    if row['relation_type'] in ['replied', 'retweeted', 'quoted']:
        tgt = row['target_username']
        if pd.notna(tgt) and tgt != '-' and tgt.strip() != '' and tgt != src and tgt not in exc and src not in exc:
            edges.append({
                'source': f'@{src}',
                'target': f'@{tgt}',
                'relation': row['relation_type'],
                'normal_text': normal_text,
                'stemm_text': stemm_text,
                'sentiment': sentiment
            })

    # === MENTIONED ===
    tgt = row.get('target_username')
    if pd.notna(row['user_mentions']) and row['user_mentions'] != '-':
        mentions = [m.replace('@', '').strip() for m in row['user_mentions'].split(';') if m.strip()]
        for m in mentions:
            if (
                m != src and 
                m != tgt and 
                m not in exc and 
                src not in exc
            ):
                edges.append({
                    'source': f'@{src}',
                    'target': f'@{m}',
                    'relation': 'mentioned',
                    'normal_text': normal_text,
                    'stemm_text': stemm_text,
                    'sentiment': sentiment
                })

edges_df = pd.DataFrame(edges, columns=['source', 'target', 'relation','normal_text','stemm_text','sentiment'])

edges_df.to_csv(f"{output_folder}/all.csv", index=False)

print("[OK] Total data:", len(df))
print("[OK] Total nodes:", pd.concat([edges_df['source'], edges_df['target']]).nunique())
print("[OK] Total edges:", len(edges_df))
print("-" * 20)
print(edges_df.info(),'\n')
display(edges_df.head(5))

[OK] Total data: 565
[OK] Total nodes: 455
[OK] Total edges: 598
--------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 598 entries, 0 to 597
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   source       598 non-null    object
 1   target       598 non-null    object
 2   relation     598 non-null    object
 3   normal_text  598 non-null    object
 4   stemm_text   598 non-null    object
 5   sentiment    598 non-null    object
dtypes: object(6)
memory usage: 28.2+ KB
None 



,source,target,relation,normal_text,stemm_text,sentiment
0,@raya_lubuk63207,@msaid_didu,retweeted,jika pasal 33 uud 45 tidak dilaksanakan dan ol...,pasal uud laksana oligarki asai,negative
1,@raya_lubuk63207,@YouTube,mentioned,jika pasal 33 uud 45 tidak dilaksanakan dan ol...,pasal uud laksana oligarki asai,negative
2,@noctiscloud28,@msaid_didu,retweeted,jika pasal 33 uud 45 tidak dilaksanakan dan ol...,pasal uud laksana oligarki asai,negative
3,@noctiscloud28,@YouTube,mentioned,jika pasal 33 uud 45 tidak dilaksanakan dan ol...,pasal uud laksana oligarki asai,negative
4,@Brutus_Istn2045,@txtpersahaman,replied,"uud pasal 33, benar tidak bunyinya begini?: bu...",uud pasal bunyi bumi air kaya alam dalam asai ...,negative


In [3]:
df = pd.read_csv(f"{output_folder}/all.csv")

relations = ['mentioned', 'replied', 'quoted', 'retweeted']

for rel in relations:
    subset = df[df['relation'] == rel][['source', 'target','normal_text','stemm_text', 'sentiment']].copy()
    if subset.empty:
        print(f"[WARNING] {rel}: kosong, dilewati.")
        print("-" * 80)
        continue

    subset = subset[subset['source'] != subset['target']]

    outpath = f"{output_folder}/{rel}.csv"
    subset.to_csv(outpath, index=False)
    print(f"[OK] {rel} nodes: {pd.concat([subset['source'], subset['target']]).nunique()}")
    print(f"[OK] {rel} edges: {len(subset)}")
    print(f"[OK] {rel}: {len(subset)} data disimpan ke {outpath}")
    print("-" * 80)

merge_rels = ['mentioned', 'retweeted']

merged_subset = df[df['relation'].isin(merge_rels)][['source', 'target', 'normal_text', 'stemm_text', 'sentiment', 'relation']].copy()

merged_subset = merged_subset[merged_subset['source'] != merged_subset['target']]

if merged_subset.empty:
    print("[WARNING] merged {merge_rels[0]}+{merge_rels[1]}: kosong, dilewati.")
    print("-" * 80)
else:
    outpath = f"{output_folder}/{merge_rels[0]}_{merge_rels[1]}.csv"
    merged_subset.to_csv(outpath, index=False)

    print(f"[OK] merged {merge_rels[0]}+{merge_rels[1]} nodes: "
          f"{pd.concat([merged_subset['source'], merged_subset['target']]).nunique()}")
    print(f"[OK] merged {merge_rels[0]}+{merge_rels[1]} edges: {len(merged_subset)}")
    print(f"[OK] merged {merge_rels[0]}+{merge_rels[1]}: {len(merged_subset)} data disimpan ke {outpath}")
    print("-" * 80)

[OK] mentioned nodes: 175
[OK] mentioned edges: 212
[OK] mentioned: 212 data disimpan ke ../../result/pasal_33_uud/tweets/sna/mentioned.csv
--------------------------------------------------------------------------------
[OK] replied nodes: 142
[OK] replied edges: 113
[OK] replied: 113 data disimpan ke ../../result/pasal_33_uud/tweets/sna/replied.csv
--------------------------------------------------------------------------------
[OK] quoted nodes: 45
[OK] quoted edges: 26
[OK] quoted: 26 data disimpan ke ../../result/pasal_33_uud/tweets/sna/quoted.csv
--------------------------------------------------------------------------------
[OK] retweeted nodes: 227
[OK] retweeted edges: 247
[OK] retweeted: 247 data disimpan ke ../../result/pasal_33_uud/tweets/sna/retweeted.csv
--------------------------------------------------------------------------------
[OK] merged mentioned+retweeted nodes: 336
[OK] merged mentioned+retweeted edges: 459
[OK] merged mentioned+retweeted: 459 data disimpan ke

In [4]:
def build_stopwords():
    download('stopwords', quiet=True)
    stopword_list = stopwords.words('indonesian')
    extra = [
        'a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z',
        'aww','awww','tu','tuh','uf','deh','dah','amp','je','rm','ufcc','ppv','of',
        'sih','eh','hm','hmm','hmmm','nya','kah','kan','kalo','kayak','doang','dll','tau',
        'ufufuf','lho','loh','lah','pon','je','pe','gs','ya','go','bro',
        'mah','lu','lo','loe','kau','ku','kuu','tau','ni','nih','gue','gua','gw',
        'kah','kahh','kak','kakk','kakak','kaka','kakkk','wkwk','wkwkwk','wkwkwkwk',
        'bang','abang','guy','guys','gaes','bawa','sok','pas','iya',
        # English stopwords
        'the','is','are','was','were','be','been','being','have','has','had',
        'do','does','did','will','would','shall','should','may','might','must',
        'can','could','it','its','an','at','by','for','from','in','into',
        'on','to','with','as','if','or','and','but','not','no','so','than',
        'that','this','then','what','which','who','whom','how','all','each',
        'every','both','few','more','most','other','some','such','only','own',
        'same','very','just','because','about','up','out','off','over','under',
        'again','further','once','here','there','when','where','why','ch',
        # Custom Sesuai Project
        'pasal','33','uud','45','1945','uud 1945','uud 45','pasal 33',
        'presiden','prabowo'
    ]
    stopword_list.extend(extra)
    return list(set(stopword_list))

# Generate Summary All

In [5]:
def process_all(
    input_file: str,
    output_dir: str,
    embedding_model
):
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(input_file)
    if df.empty:
        print("[WARNING] File kosong, tidak ada data yang diproses.")
        return

    print(f"\nMulai olah seluruh data | {len(df)} baris data")

    # ==== DEGREE ==== #
    edges_unique = df[['source', 'target']].drop_duplicates()
    outdegree = edges_unique['source'].value_counts()
    indegree = edges_unique['target'].value_counts()
    degree = (
        outdegree.add(indegree, fill_value=0)
        .sort_values(ascending=False)
    )

    # ==== WEIGHTED DEGREE ==== #
    edges_weighted = df.groupby(['source', 'target'], as_index=False).size().rename(columns={'size': 'weight'})
    outdegree_w = edges_weighted.groupby('source')['weight'].sum()
    indegree_w = edges_weighted.groupby('target')['weight'].sum()
    degree_w = outdegree_w.add(indegree_w, fill_value=0).sort_values(ascending=False)

    top_users = indegree_w.sort_values(ascending=False).head(5).index.tolist()
    print(f"Top {len(top_users)} user berdasarkan in-degree:\n{top_users}\n")

    stop_words = build_stopwords()
    topic_summary_list = []

    for user in tqdm(top_users):
        df_user = df[
            (df['relation'].isin(['mentioned', 'replied', 'quoted'])) &
            ((df['target'] == user) | (df['source'] == user))
        ]
        normal_docs = df_user['normal_text'].dropna().tolist()
        stemm_docs = df_user['stemm_text'].dropna().tolist()

        # Dedup dokumen (1 tweet bisa muncul banyak kali di edge berbeda)
        normal_docs = list(dict.fromkeys(normal_docs))
        stemm_docs = list(dict.fromkeys(stemm_docs))

        if len(normal_docs) < 15:
            print(f"[WARNING] Skip {user} (dokumen terlalu sedikit: {len(normal_docs)})")
            continue

        print(f"Proses {user}: {len(normal_docs)} tweet | Degree {int(degree.get(user, 0))} | In: {int(indegree.get(user, 0))} | Out: {int(outdegree.get(user, 0))}")

        sentiment_counts = df_user['sentiment'].value_counts(normalize=True).to_dict()
        dominant_sentiment = max(sentiment_counts, key=sentiment_counts.get)
        sentiment_readable = " | ".join([f"{k.capitalize()}: {v*100:.1f}%" for k, v in sentiment_counts.items()])

        # === Topic Modeling (stemm text) → keyword extraction === #
        min_topic_size = max(5, min(15, len(stemm_docs)//5))
        vectorizer = CountVectorizer(stop_words=stop_words, ngram_range=(1, 1))
        stemm_topic_model = BERTopic(
            embedding_model=embedding_model,
            vectorizer_model=vectorizer,
            min_topic_size=min_topic_size,
            verbose=False,
            nr_topics=min(10, max(2, len(stemm_docs)//3))
        )

        topics, probs = stemm_topic_model.fit_transform(stemm_docs)
        topic_info = stemm_topic_model.get_topic_info()

        keywords_all = []
        for _, row in topic_info.iterrows():
            if row['Topic'] != -1 and isinstance(row['Name'], str):
                clean_name = re.sub(r"^\d+_", "", row['Name']).strip()
                words = [w for w in clean_name.split('_') if len(w) > 2 and w not in stop_words]
                keywords_all.extend(words)

        if not keywords_all:
            all_words = []
            for doc in stemm_docs:
                for w in doc.split():
                    if len(w) > 2 and w not in stop_words:
                        all_words.append(w)
            counter = Counter(all_words)
        else:
            counter = Counter(keywords_all)

        top_words = [w for w, _ in counter.most_common(5)]

        # === Topic Modeling (normal text) → representative docs === #
        min_topic_size = max(5, min(15, len(normal_docs)//5))
        vectorizer = CountVectorizer(stop_words=stop_words, ngram_range=(1, 2))
        normal_topic_model = BERTopic(
            embedding_model=embedding_model,
            vectorizer_model=vectorizer,
            min_topic_size=min_topic_size,
            verbose=False,
            nr_topics=min(10, max(2, len(normal_docs)//3))
        )

        normal_topic_model.fit_transform(normal_docs)

        try:
            rep_docs_all = []
            rep_docs = normal_topic_model.get_representative_docs()
            for k, v in rep_docs.items():
                if k != -1:
                    rep_docs_all.extend(v[:3])
        except Exception:
            rep_docs_all = []

        if not rep_docs_all:
            rep_docs_all = normal_docs[:5]

        rep_docs_all = list(dict.fromkeys(rep_docs_all))

        relations_used = df_user['relation'].unique()
        relation_scope = ', '.join(sorted(set(relations_used)))

        topic_summary_list.append({
            'user': user,
            'top_keywords': ' | '.join(sorted(set(top_words))),
            'narasi': ' ... '.join(rep_docs_all[:10]),
            'relation_scope': relation_scope,
            'dominant_sentiment': dominant_sentiment,
            'sentiment_distribution': sentiment_readable,
            'tweet_count': len(normal_docs),
            'degree': int(degree.get(user, 0)),
            'degree_distribution': f"in: {int(indegree.get(user, 0))} | out: {int(outdegree.get(user, 0))}",
            'weighted_degree': float(degree_w.get(user, 0)),
            'w_degree_distribution': f"in: {float(indegree_w.get(user, 0))} | out: {float(outdegree_w.get(user, 0))}"
        })

    # === Simpan hasil === #
    result_df = pd.DataFrame(topic_summary_list)
    result_df = result_df.sort_values(by='tweet_count', ascending=False)
    outpath = f"{output_dir}/all_summary.csv"
    result_df.to_csv(outpath, index=False, encoding='utf-8-sig')

    print(f"\n[OK] Selesai olah all.csv — hasil disimpan ke: {outpath}")
    return result_df
    

process_all(
    input_file=f"{output_folder}/all.csv",
    output_dir=output_folder,
    embedding_model=embedding_model
)


Mulai olah seluruh data | 598 baris data
Top 5 user berdasarkan in-degree:
['@Boediantar4', '@msaid_didu', '@YouTube', '@prabowo', '@fadlizon']



  0%|          | 0/5 [00:00<?, ?it/s]

[WARNING] Skip @Boediantar4 (dokumen terlalu sedikit: 4)
[WARNING] Skip @msaid_didu (dokumen terlalu sedikit: 7)
[WARNING] Skip @YouTube (dokumen terlalu sedikit: 1)
Proses @prabowo: 47 tweet | Degree 38 | In: 38 | Out: 0


 80%|████████  | 4/5 [00:38<00:09,  9.74s/it]

Proses @fadlizon: 16 tweet | Degree 34 | In: 31 | Out: 3


100%|██████████| 5/5 [00:44<00:00,  8.95s/it]


[OK] Selesai olah all.csv — hasil disimpan ke: ../../result/pasal_33_uud/tweets/sna/all_summary.csv


,user,top_keywords,narasi,relation_scope,dominant_sentiment,sentiment_distribution,tweet_count,degree,degree_distribution,weighted_degree,w_degree_distribution
0,@prabowo,ekonomi | ekspor | indonesia | negara | rakyat,pidato presiden pada rapat paripurna dpr ri ke...,mentioned,negative,Negative: 59.3% | Neutral: 33.3% | Positive: 7.4%,47,38,in: 38 | out: 0,54.0,in: 54.0 | out: 0.0
1,@fadlizon,ekonomi | milik | negara | program | rakyat,bagaimana menilai apakah suatu program benar-b...,"mentioned, quoted, replied",negative,Negative: 70.6% | Neutral: 29.4%,16,34,in: 31 | out: 3,36.0,in: 33.0 | out: 3.0


# Generate Summary based on relation

In [ ]:
def process_relation(
    relation: str,
    input_dir: str,
    output_dir: str,
    embedding_model
):
    os.makedirs(output_dir, exist_ok=True)

    file_path = f"{input_dir}/{relation}.csv"
    df = pd.read_csv(file_path)

    if df.empty:
        print(f"[WARNING] File kosong: {relation}")
        return

    # ==== DEGREE ==== #
    edges_unique = df[['source', 'target']].drop_duplicates()
    outdegree = edges_unique['source'].value_counts()
    indegree = edges_unique['target'].value_counts()
    degree = (
        outdegree.add(indegree, fill_value=0)
        .sort_values(ascending=False)
    )

    top_users = indegree.head(5).index.tolist()

    print(f"\nMulai olah relasi: {relation.upper()} | {len(df)} baris data")

    stop_words = build_stopwords()
    topic_summary_list = []

    for user in tqdm(top_users):
        df_user = df[(df['target'] == user) | (df['source'] == user)]
        normal_docs = df_user['normal_text'].dropna().tolist()
        stemm_docs = df_user['stemm_text'].dropna().tolist()

        # Dedup dokumen (1 tweet bisa muncul banyak kali di edge berbeda)
        normal_docs = list(dict.fromkeys(normal_docs))
        stemm_docs = list(dict.fromkeys(stemm_docs))

        if len(normal_docs) < 15:
            print(f"[WARNING] Skip {user} (dokumen terlalu sedikit: {len(normal_docs)})")
            continue

        print(f"\nProses {user}: {len(normal_docs)} tweet | Degree {int(degree.get(user, 0))} | In-degree {int(indegree.get(user, 0))} | Out-degree {int(outdegree.get(user, 0))}")

        sentiment_counts = df_user['sentiment'].value_counts(normalize=True).to_dict()
        dominant_sentiment = max(sentiment_counts, key=sentiment_counts.get)
        sentiment_readable = " | ".join([f"{k.capitalize()}: {v*100:.1f}%" for k, v in sentiment_counts.items()])

        # === Topic Modeling (stemm text) → keyword extraction === #
        min_topic_size = max(5, min(15, len(stemm_docs)//5))
        vectorizer = CountVectorizer(stop_words=stop_words, ngram_range=(1, 1))
        stemm_topic_model = BERTopic(
            embedding_model=embedding_model,
            vectorizer_model=vectorizer,
            min_topic_size=min_topic_size,
            verbose=False,
            nr_topics=min(10, max(2, len(stemm_docs)//3))
        )

        topics, probs = stemm_topic_model.fit_transform(stemm_docs)
        topic_info = stemm_topic_model.get_topic_info()

        keywords_all = []
        for _, row in topic_info.iterrows():
            if row['Topic'] != -1 and isinstance(row['Name'], str):
                clean_name = re.sub(r"^\d+_", "", row['Name']).strip()
                words = [w for w in clean_name.split('_') if len(w) > 2 and w not in stop_words]
                keywords_all.extend(words)

        if not keywords_all:
            all_words = []
            for doc in stemm_docs:
                for w in doc.split():
                    if len(w) > 2 and w not in stop_words:
                        all_words.append(w)
            counter = Counter(all_words)
        else:
            counter = Counter(keywords_all)

        top_words = [w for w, _ in counter.most_common(5)]

        # === Topic Modeling (normal text) → representative docs === #
        min_topic_size = max(5, min(15, len(normal_docs)//5))
        vectorizer = CountVectorizer(stop_words=stop_words, ngram_range=(1, 2))
        normal_topic_model = BERTopic(
            embedding_model=embedding_model,
            vectorizer_model=vectorizer,
            min_topic_size=min_topic_size,
            verbose=False,
            nr_topics=min(10, max(2, len(normal_docs)//3))
        )

        normal_topic_model.fit_transform(normal_docs)

        try:
            rep_docs_all = []
            rep_docs = normal_topic_model.get_representative_docs()
            for k, v in rep_docs.items():
                if k != -1:
                    rep_docs_all.extend(v[:3])

        except Exception:
            rep_docs_all = []

        if not rep_docs_all:
            rep_docs_all = normal_docs[:5]

        rep_docs_all = list(dict.fromkeys(rep_docs_all))

        topic_summary_list.append({
            'user': user,
            'top_keywords': ' | '.join(sorted(set(top_words))),
            'narasi': ' ... '.join(rep_docs_all[:10]),
            'relation_scope': relation,
            'dominant_sentiment': dominant_sentiment,
            'sentiment_distribution': sentiment_readable,
            'tweet_count': len(normal_docs),
            'degree': int(degree.get(user, 0)),
            'degree_distribution': f"in: {int(indegree.get(user, 0))} | out: {int(outdegree.get(user, 0))}"
        })

    # ==== Simpan Hasil ==== #
    result_df = pd.DataFrame(topic_summary_list)
    result_df = result_df.sort_values(by='tweet_count', ascending=False)
    outpath = f"{output_dir}/{relation}_summary.csv"
    result_df.to_csv(outpath, index=False, encoding='utf-8-sig')

    print(f"\n[OK] {relation.upper()} selesai — hasil disimpan ke: {outpath}")
    return result_df

process_relation(
    relation="quoted", # Disesuaikan dengan relasi yang ingin diproses : mentioned | replied | quoted | retweeted
    input_dir=output_folder,
    output_dir=output_folder,
    embedding_model=embedding_model
)

# Topic Modeling

In [7]:
def plot_topic_sentiment_barchart(df, topics, topic_model, output_dir):
    plot_dir = os.path.join(output_dir, "plot")
    os.makedirs(plot_dir, exist_ok=True)

    df_local = df.copy()
    df_local["topic"] = topics

    if "sentiment" not in df_local.columns:
        print("[WARNING] Kolom 'sentiment' tidak ditemukan. Skip.")
        return

    stop_words = build_stopwords()
    sentiments = ["negative", "neutral", "positive"]

    for sent in sentiments:
        sub = df_local[df_local["sentiment"] == sent]
        if sub.empty:
            print(f"[INFO] Tidak ada dokumen untuk sentiment '{sent}', skip.")
            continue

        topics_dict = {}

        # Hitung frekuensi kata dari normal_text (lebih readable)
        docs = sub["normal_text"].fillna("").astype(str).tolist()
        vect = CountVectorizer(lowercase=True, stop_words=stop_words, ngram_range=(1, 2), token_pattern=r"(?u)\b\w+\b")

        try:
            vect.fit(docs)
            X = vect.transform(docs)
            vocab = vect.vocabulary_
        except Exception:
            X = None
            vocab = {}

        for tid in sorted(topic_model.get_topics().keys()):
            if tid == -1:
                continue

            top_terms = topic_model.get_topic(tid)[:5]
            words = [w for w, _ in top_terms]

            if not words:
                topics_dict[tid] = {"words": [], "values": []}
                continue

            values = []
            if X is not None and vocab:
                for w in words:
                    if w in vocab:
                        col_idx = vocab[w]
                        freq = int(X[:, col_idx].sum())
                    else:
                        freq = 0
                    values.append(freq)
            else:
                values = [0] * len(words)

            # fallback jika semua nol → pakai weight
            if all(v == 0 for v in values):
                values = [float(v) for _, v in top_terms]

            topics_dict[tid] = {"words": words, "values": values}

        if not topics_dict:
            print(f"[INFO] Tidak ada topik untuk sentiment '{sent}', skip plot.")
            continue

        out_path = os.path.join(plot_dir, f"barchart_sentiment_{sent}.png")
        plot_multi_topic_barchart(
            topics_dict,
            f"Top-5 Kata per Topik (sentiment: {sent})",
            out_path
        )
        print(f"[OK] Multi-barchart sentiment '{sent}' → {out_path}")


def plot_multi_topic_barchart(topics_dict, title, output_path):
    n = len(topics_dict)
    cols = 4
    rows = math.ceil(n / cols) if n > 0 else 1

    plt.figure(figsize=(cols * 4.2, rows * 3.0))
    plt.suptitle(title, fontsize=16)

    for idx, (tid, info) in enumerate(topics_dict.items(), start=1):
        plt.subplot(rows, cols, idx)

        words = info.get("words", [])
        values = info.get("values", [])

        if not words:
            plt.title(f"Topic {tid} (no terms)", fontsize=10)
            plt.axis("off")
            continue

        # reverse biar kata terbesar di atas
        words_rev = words[::-1]
        vals_rev = values[::-1]

        plt.barh(words_rev, vals_rev)
        plt.title(f"Topic {tid}", fontsize=10)
        plt.xticks(fontsize=8)
        plt.yticks(fontsize=8)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(output_path, dpi=300)
    plt.close()

In [8]:
def global_topic_model(
    input_file: str,
    output_dir: str,
    embedding_model
):
    os.makedirs(output_dir, exist_ok=True)
    plot_dir = os.path.join(output_dir, "plot")
    os.makedirs(plot_dir, exist_ok=True)

    df = pd.read_csv(input_file)

    # Dedup + drop NaN agar topics & df selaras
    df_dedup = df.drop_duplicates(subset=['normal_text']).dropna(subset=['normal_text', 'stemm_text']).reset_index(drop=True)
    docs = df_dedup['normal_text'].tolist()
    print(f"Jumlah dokumen: {len(docs)} (dari {len(df)} edges, setelah dedup)")

    stop_words = build_stopwords()

    # Pakai normal_text + bigram agar plot menampilkan frasa bermakna
    vectorizer = CountVectorizer(
        ngram_range=(1, 2),
        stop_words=stop_words
    )

    topic_model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer,
        nr_topics=5,
        top_n_words=5,
        min_topic_size=5,
        verbose=False
    )

    topics, probs = topic_model.fit_transform(docs)
    topic_info = topic_model.get_topic_info()

    # clean keyword
    topic_keywords = []
    for _, row in topic_info.iterrows():
        tid = row["Topic"]
        if tid == -1 or not isinstance(row["Name"], str):
            topic_keywords.append((tid, []))
            continue

        clean = re.sub(r"^\d+_", "", row["Name"]).strip()
        words = [
            w for w in clean.split("_")
            if len(w) > 2 and w not in stop_words
        ]
        topic_keywords.append((tid, words))

    kw_df = pd.DataFrame(topic_keywords, columns=["Topic", "top_keywords"])
    topic_info = topic_info.merge(kw_df, on="Topic", how="left")

    csv_out = f"{output_dir}/global_topics.csv"
    topic_info.to_csv(csv_out, index=False, encoding='utf-8-sig')
    print(f"[OK] Topic CSV disimpan → {csv_out}")

    # === Global barchart === #
    try:
        topics_words = {}

        for tid in sorted(topic_model.get_topics().keys()):
            if tid == -1:
                continue

            top_terms = topic_model.get_topic(tid)[:5]
            if not top_terms:
                continue

            words = [w for w, _ in top_terms]
            values = [float(v) for _, v in top_terms]

            topics_words[tid] = {"words": words, "values": values}

        out_path = os.path.join(plot_dir, "barchart_global.png")
        plot_multi_topic_barchart(
            topics_words,
            "Global: Top-5 Kata/Frasa per Topik",
            out_path
        )
        print(f"[OK] Global multi-barchart → {out_path}")

    except Exception as e:
        print(f"[WARNING] Gagal plot global: {e}")

    # === Sentiment charts (df_dedup & topics sudah selaras) === #
    plot_topic_sentiment_barchart(df_dedup, topics, topic_model, output_dir)

    return topic_info

global_topic_model(
    input_file=f"{output_folder}/all.csv",
    output_dir=output_folder,
    embedding_model=embedding_model
)

Jumlah dokumen: 169 (dari 598 edges, setelah dedup)
[OK] Topic CSV disimpan → ../../result/pasal_33_uud/tweets/sna/global_topics.csv
[OK] Global multi-barchart → ../../result/pasal_33_uud/tweets/sna\plot\barchart_global.png
[OK] Multi-barchart sentiment 'negative' → ../../result/pasal_33_uud/tweets/sna\plot\barchart_sentiment_negative.png
[OK] Multi-barchart sentiment 'neutral' → ../../result/pasal_33_uud/tweets/sna\plot\barchart_sentiment_neutral.png
[OK] Multi-barchart sentiment 'positive' → ../../result/pasal_33_uud/tweets/sna\plot\barchart_sentiment_positive.png


,Topic,Count,Name,Representation,Representative_Docs,top_keywords
0,-1,40,-1_ekonomi_rakyat_negara_politik,"[ekonomi, rakyat, negara, politik, indonesia]",[pidato presiden pada rapat paripurna dpr ri k...,[]
1,0,58,0_negara_rakyat_ayat_dikuasai,"[negara, rakyat, ayat, dikuasai, alam]",[wajib dikuasai negara!! pasal 33 ayat 3 uud 1...,"[negara, rakyat, ayat, dikuasai]"
2,1,32,1_amanat_sda_garong_ayat,"[amanat, sda, garong, ayat, menjalankan]",[sikap psi jelas membawa dan melanjutkan misi ...,"[amanat, sda, garong, ayat]"
3,2,29,2_indonesia_negara_ekspor_alam,"[indonesia, negara, ekspor, alam, ekonomi]",[qodari: presiden perkuat pengawasan ekspor de...,"[indonesia, negara, ekspor, alam]"
4,3,10,3_kebijakan_akun_ekspor_koar,"[kebijakan, akun, ekspor, koar, introspeksi]",[seharusnya seorang presiden harus lebih memen...,"[kebijakan, akun, ekspor, koar]"
